# KOCOH 제안모델 - BERT 문맥 결합 모델 최종 버전

최종 성능이 가장 잘 나왔던 설정을 기준으로 다시 정리한 노트북입니다.

## 최종 설정

```text
모델: klue/bert-base
입력: context + comment
split: context 기준 group split
MAX_LENGTH: 256
EPOCHS: 5
LR: 2e-5
class weight: [1.0, 1.3]
threshold: 0.5
scheduler: 사용하지 않음
```

기존 최고 결과:

```text
Accuracy : 0.8787
Precision: 0.7867
Recall   : 0.7024
F1-score : 0.7421
```

실행 전 Colab에 `KOCOH_v.2.csv`를 업로드하세요.


## 1. 라이브러리 설치

In [1]:
!pip install transformers scikit-learn -q

## 2. 기본 설정

In [2]:
import random
import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModelForSequenceClassification

SEED = 42
MODEL_NAME = "klue/bert-base"

MAX_LENGTH = 256
BATCH_SIZE = 16
EPOCHS = 5
LR = 2e-5
THRESHOLD = 0.5

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("사용 장치:", device)
print("MODEL_NAME:", MODEL_NAME)
print("MAX_LENGTH:", MAX_LENGTH)
print("BATCH_SIZE:", BATCH_SIZE)
print("EPOCHS:", EPOCHS)
print("LR:", LR)
print("THRESHOLD:", THRESHOLD)


사용 장치: cuda
MODEL_NAME: klue/bert-base
MAX_LENGTH: 256
BATCH_SIZE: 16
EPOCHS: 5
LR: 2e-05
THRESHOLD: 0.5


## 3. 원본 데이터 불러오기

원본 `KOCOH_v.2.csv`에서 기존 전처리 파일과 같은 형식으로 컬럼명을 맞춥니다.

```text
Context → context
Comment → comment
Hate speech → label
Type → type
```

라벨 의미:

```text
0 = 비혐오
1 = 혐오
```


In [8]:
DATA_PATH = "/content/KOCOH_v.2.csv"

raw_df = pd.read_csv(DATA_PATH)

print("원본 데이터 크기:", raw_df.shape)
print("원본 컬럼:", raw_df.columns.tolist())
display(raw_df.head())

df = raw_df.rename(columns={
    "Context": "context",
    "Comment": "comment",
    "Hate speech": "label",
    "Type": "type"
})

if "type" not in df.columns:
    df["type"] = -1

df = df[["context", "comment", "label", "type"]].copy()

df = df.dropna(subset=["context", "comment", "label"]).reset_index(drop=True)
df["context"] = df["context"].astype(str)
df["comment"] = df["comment"].astype(str)
df["label"] = df["label"].astype(int)

print("정리 후 데이터 크기:", df.shape)
print("라벨 분포")
print(df["label"].value_counts())
print("라벨 비율")
print(df["label"].value_counts(normalize=True))
print("고유 context 수:", df["context"].nunique())

display(df.head())


원본 데이터 크기: (3000, 18)
원본 컬럼: ['Index', 'Set', 'Type', 'Date', 'Source', 'Link', 'Title', 'Context', 'Comment', 'Hate speech', 'Counter speech', 'Gender', 'Disability', 'Race/Nationality', 'Region (Korea)', 'Age', 'Profanity', 'Ikiyano-style']


,Index,Set,Type,Date,Source,Link,Title,Context,Comment,Hate speech,Counter speech,Gender,Disability,Race/Nationality,Region (Korea),Age,Profanity,Ikiyano-style
0,1,1,1,2024-06-02,디시인사이드,https://gall.dcinside.com/board/view/?id=dcbes...,싱글벙글 이제부터 여성 안 뽑는 '여혐업체' 박제한다....jpg,여성 고용이 미흡해 정부가 불이익을 준 기업의 이름이 공개되었다.,미래가 밝은 회사이니 투자하라는 뜻에서 샤라웃 해주는 거임,1,0,1,0,0,0,0,0,0
1,2,1,1,2024-06-02,디시인사이드,https://gall.dcinside.com/board/view/?id=dcbes...,싱글벙글 이제부터 여성 안 뽑는 '여혐업체' 박제한다....jpg,여성 고용이 미흡해 정부가 불이익을 준 기업의 이름이 공개되었다.,쉐보레 차 좋은데엔 이유가 있었노,1,0,1,0,0,0,0,0,1
2,3,1,1,2024-06-02,디시인사이드,https://gall.dcinside.com/board/view/?id=dcbes...,싱글벙글 이제부터 여성 안 뽑는 '여혐업체' 박제한다....jpg,여성 고용이 미흡해 정부가 불이익을 준 기업의 이름이 공개되었다.,풀매수 간다,1,0,1,0,0,0,0,0,0
3,4,1,1,2024-06-02,디시인사이드,https://gall.dcinside.com/board/view/?id=dcbes...,싱글벙글 이제부터 여성 안 뽑는 '여혐업체' 박제한다....jpg,여성 고용이 미흡해 정부가 불이익을 준 기업의 이름이 공개되었다.,옳게된 기업,1,0,1,0,0,0,0,0,0
4,5,1,1,2024-06-02,디시인사이드,https://gall.dcinside.com/board/view/?id=dcbes...,싱글벙글 이제부터 여성 안 뽑는 '여혐업체' 박제한다....jpg,여성 고용이 미흡해 정부가 불이익을 준 기업의 이름이 공개되었다.,그니까 저기에 주식투자 하면 된다는거지?,1,0,1,0,0,0,0,0,0


정리 후 데이터 크기: (3000, 4)
라벨 분포
label
0    2236
1     764
Name: count, dtype: int64
라벨 비율
label
0    0.745333
1    0.254667
Name: proportion, dtype: float64
고유 context 수: 282


,context,comment,label,type
0,여성 고용이 미흡해 정부가 불이익을 준 기업의 이름이 공개되었다.,미래가 밝은 회사이니 투자하라는 뜻에서 샤라웃 해주는 거임,1,1
1,여성 고용이 미흡해 정부가 불이익을 준 기업의 이름이 공개되었다.,쉐보레 차 좋은데엔 이유가 있었노,1,1
2,여성 고용이 미흡해 정부가 불이익을 준 기업의 이름이 공개되었다.,풀매수 간다,1,1
3,여성 고용이 미흡해 정부가 불이익을 준 기업의 이름이 공개되었다.,옳게된 기업,1,1
4,여성 고용이 미흡해 정부가 불이익을 준 기업의 이름이 공개되었다.,그니까 저기에 주식투자 하면 된다는거지?,1,1


## 4. Context 기준 Group Split

제안모델은 `context + comment`를 함께 입력으로 사용하므로, 같은 `context`가 train/test에 동시에 들어가면 평가 성능이 과대평가될 수 있습니다.

따라서 동일한 `context`가 서로 다른 split에 들어가지 않도록 group split을 적용합니다.


In [10]:
def split_score(train_df, valid_df, test_df, full_df):
    total = len(full_df)
    target_label_ratio = full_df["label"].mean()

    size_score = (
        abs(len(train_df) / total - 0.8)
        + abs(len(valid_df) / total - 0.1)
        + abs(len(test_df) / total - 0.1)
    )

    label_score = (
        abs(train_df["label"].mean() - target_label_ratio)
        + abs(valid_df["label"].mean() - target_label_ratio)
        + abs(test_df["label"].mean() - target_label_ratio)
    )

    return size_score + 5 * label_score


def make_context_group_split(full_df, n_trials=500, seed=42):
    best_result = None
    best_score = float("inf")

    for i in range(n_trials):
        current_seed = seed + i

        gss1 = GroupShuffleSplit(
            n_splits=1,
            test_size=0.2,
            random_state=current_seed
        )

        train_idx, temp_idx = next(
            gss1.split(
                full_df,
                y=full_df["label"],
                groups=full_df["context"]
            )
        )

        train_df = full_df.iloc[train_idx].reset_index(drop=True)
        temp_df = full_df.iloc[temp_idx].reset_index(drop=True)

        gss2 = GroupShuffleSplit(
            n_splits=1,
            test_size=0.5,
            random_state=current_seed + 10000
        )

        valid_idx, test_idx = next(
            gss2.split(
                temp_df,
                y=temp_df["label"],
                groups=temp_df["context"]
            )
        )

        valid_df = temp_df.iloc[valid_idx].reset_index(drop=True)
        test_df = temp_df.iloc[test_idx].reset_index(drop=True)

        if train_df["label"].nunique() < 2:
            continue
        if valid_df["label"].nunique() < 2:
            continue
        if test_df["label"].nunique() < 2:
            continue

        score = split_score(train_df, valid_df, test_df, full_df)

        if score < best_score:
            best_score = score
            best_result = (train_df, valid_df, test_df, current_seed, score)

    if best_result is None:
        raise ValueError("적절한 split을 찾지 못했습니다. n_trials 값을 늘려보세요.")

    return best_result


train_df, valid_df, test_df, selected_seed, best_score = make_context_group_split(
    df,
    n_trials=500,
    seed=SEED
)

print("선택된 split seed:", selected_seed)
print("split score:", best_score)

print("데이터 크기")
print("train:", train_df.shape)
print("valid:", valid_df.shape)
print("test :", test_df.shape)

print("train 라벨 분포")
print(train_df["label"].value_counts())
print(train_df["label"].value_counts(normalize=True))

print("valid 라벨 분포")
print(valid_df["label"].value_counts())
print(valid_df["label"].value_counts(normalize=True))

print("test 라벨 분포")
print(test_df["label"].value_counts())
print(test_df["label"].value_counts(normalize=True))


선택된 split seed: 503
split score: 0.12414660012138615
데이터 크기
train: (2408, 4)
valid: (254, 4)
test : (338, 4)
train 라벨 분포
label
0    1790
1     618
Name: count, dtype: int64
label
0    0.743355
1    0.256645
Name: proportion, dtype: float64
valid 라벨 분포
label
0    192
1     62
Name: count, dtype: int64
label
0    0.755906
1    0.244094
Name: proportion, dtype: float64
test 라벨 분포
label
0    254
1     84
Name: count, dtype: int64
label
0    0.751479
1    0.248521
Name: proportion, dtype: float64


## 5. Context 중복 확인 및 split 파일 저장

In [11]:
def check_context_overlap(train_df, valid_df, test_df):
    train_contexts = set(train_df["context"])
    valid_contexts = set(valid_df["context"])
    test_contexts = set(test_df["context"])

    train_valid_overlap = train_contexts & valid_contexts
    train_test_overlap = train_contexts & test_contexts
    valid_test_overlap = valid_contexts & test_contexts

    print("train-valid context overlap:", len(train_valid_overlap))
    print("train-test  context overlap:", len(train_test_overlap))
    print("valid-test  context overlap:", len(valid_test_overlap))

    assert len(train_valid_overlap) == 0
    assert len(train_test_overlap) == 0
    assert len(valid_test_overlap) == 0

    print("확인 완료: train/valid/test 사이에 겹치는 context가 없습니다.")


check_context_overlap(train_df, valid_df, test_df)

train_df.to_csv("/content/team_train_context_split.csv", index=False, encoding="utf-8-sig")
valid_df.to_csv("/content/team_valid_context_split.csv", index=False, encoding="utf-8-sig")
test_df.to_csv("/content/team_test_context_split.csv", index=False, encoding="utf-8-sig")

print("저장 완료")
print("/content/team_train_context_split.csv")
print("/content/team_valid_context_split.csv")
print("/content/team_test_context_split.csv")


train-valid context overlap: 0
train-test  context overlap: 0
valid-test  context overlap: 0
확인 완료: train/valid/test 사이에 겹치는 context가 없습니다.
저장 완료
/content/team_train_context_split.csv
/content/team_valid_context_split.csv
/content/team_test_context_split.csv


## 6. MAX_LENGTH 확인

최종 모델에서는 `MAX_LENGTH=256`을 사용합니다. 토큰 길이 분포를 확인하여 256 초과 비율이 매우 낮은지 확인합니다.


In [12]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

all_df = pd.concat([train_df, valid_df, test_df], ignore_index=True)

def get_pair_token_length(row):
    encoding = tokenizer(
        str(row["context"]),
        str(row["comment"]),
        add_special_tokens=True,
        truncation=False
    )
    return len(encoding["input_ids"])

token_lengths = []
for _, row in all_df.iterrows():
    token_lengths.append(get_pair_token_length(row))

all_df["token_length"] = token_lengths

print("토큰 길이 통계")
print(all_df["token_length"].describe())

print("분위수")
print(all_df["token_length"].quantile([0.5, 0.75, 0.9, 0.95, 0.99]))

print("MAX_LENGTH별 잘리는 샘플 비율")
for max_len in [128, 256, 384, 512]:
    truncated_count = (all_df["token_length"] > max_len).sum()
    truncated_ratio = truncated_count / len(all_df) * 100

    print(f"MAX_LENGTH={max_len}")
    print(f"잘리는 샘플 수: {truncated_count}개 / {len(all_df)}개")
    print(f"잘리는 비율: {truncated_ratio:.2f}%")
    print("-" * 40)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/425 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/289 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/248k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/495k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

토큰 길이 통계
count    3000.000000
mean       38.682667
std        16.414011
min        13.000000
25%        29.000000
50%        36.000000
75%        45.000000
max       305.000000
Name: token_length, dtype: float64
분위수
0.50    36.00
0.75    45.00
0.90    56.00
0.95    65.00
0.99    90.02
Name: token_length, dtype: float64
MAX_LENGTH별 잘리는 샘플 비율
MAX_LENGTH=128
잘리는 샘플 수: 8개 / 3000개
잘리는 비율: 0.27%
----------------------------------------
MAX_LENGTH=256
잘리는 샘플 수: 2개 / 3000개
잘리는 비율: 0.07%
----------------------------------------
MAX_LENGTH=384
잘리는 샘플 수: 0개 / 3000개
잘리는 비율: 0.00%
----------------------------------------
MAX_LENGTH=512
잘리는 샘플 수: 0개 / 3000개
잘리는 비율: 0.00%
----------------------------------------


## 7. Dataset 구성

BERT tokenizer에 `context`와 `comment`를 문장쌍 형태로 입력합니다.

```python
tokenizer(context, comment)
```

BERT 입력 구조:

```text
[CLS] context [SEP] comment [SEP]
```


In [13]:
class KocohDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length=256):
        self.contexts = dataframe["context"].astype(str).tolist()
        self.comments = dataframe["comment"].astype(str).tolist()
        self.labels = dataframe["label"].astype(int).tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.contexts[idx],
            self.comments[idx],
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )

        item = {key: value.squeeze(0) for key, value in encoding.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)

        return item


## 8. 평가 함수

In [14]:
def evaluate_model(model, dataloader, loss_fn, threshold=0.5):
    model.eval()

    total_loss = 0
    all_preds = []
    all_labels = []
    all_hate_probs = []

    with torch.no_grad():
        for batch in dataloader:
            labels = batch["labels"].to(device)

            model_inputs = {
                key: value.to(device)
                for key, value in batch.items()
                if key != "labels"
            }

            outputs = model(**model_inputs)
            logits = outputs.logits
            loss = loss_fn(logits, labels)

            total_loss += loss.item()

            probs = torch.softmax(logits, dim=1)
            hate_probs = probs[:, 1]
            preds = (hate_probs >= threshold).long()

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_hate_probs.extend(hate_probs.cpu().numpy())

    avg_loss = total_loss / len(dataloader)

    accuracy = accuracy_score(all_labels, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels,
        all_preds,
        average="binary",
        zero_division=0
    )

    return {
        "loss": avg_loss,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "y_true": all_labels,
        "y_pred": all_preds,
        "hate_probs": all_hate_probs
    }


## 9. BERT 문맥 결합 모델 학습

최종 최고 성능 설정:

```text
class_weights = [1.0, 1.3]
LR = 2e-5
EPOCHS = 5
MAX_LENGTH = 256
threshold = 0.5
scheduler 미사용
```


In [15]:
OUTPUT_NAME = "BERT_context+comment_group_split_weak_class_weight_1.3"

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

model.to(device)

train_dataset = KocohDataset(train_df, tokenizer, MAX_LENGTH)
valid_dataset = KocohDataset(valid_df, tokenizer, MAX_LENGTH)
test_dataset = KocohDataset(test_df, tokenizer, MAX_LENGTH)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# 최종 최고 성능 설정: 약한 class weight 직접 적용
class_weights = torch.tensor([1.0, 1.3], dtype=torch.float).to(device)

loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights)
optimizer = AdamW(model.parameters(), lr=LR)

print("Class weights:", class_weights)


model.safetensors:   0%|          | 0.00/445M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: klue/bert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on you

Class weights: tensor([1.0000, 1.3000], device='cuda:0')


In [17]:
best_valid_f1 = -1
best_state_dict = None
history = []

for epoch in range(EPOCHS):
    model.train()
    total_train_loss = 0

    for step, batch in enumerate(train_loader, start=1):
        optimizer.zero_grad()

        labels = batch["labels"].to(device)

        model_inputs = {
            key: value.to(device)
            for key, value in batch.items()
            if key != "labels"
        }

        outputs = model(**model_inputs)
        logits = outputs.logits
        loss = loss_fn(logits, labels)

        total_train_loss += loss.item()

        loss.backward()
        optimizer.step()

        if step % 50 == 0:
            print(
                f"Epoch {epoch+1}/{EPOCHS} | "
                f"Step {step}/{len(train_loader)} | "
                f"Loss: {loss.item():.4f}"
            )

    avg_train_loss = total_train_loss / len(train_loader)
    valid_result = evaluate_model(model, valid_loader, loss_fn, threshold=THRESHOLD)

    history.append({
        "epoch": epoch + 1,
        "train_loss": avg_train_loss,
        "valid_loss": valid_result["loss"],
        "valid_accuracy": valid_result["accuracy"],
        "valid_precision": valid_result["precision"],
        "valid_recall": valid_result["recall"],
        "valid_f1": valid_result["f1"]
    })

    print("" + "=" * 70)
    print(f"Epoch {epoch+1}/{EPOCHS}")
    print("=" * 70)
    print(f"Train Loss     : {avg_train_loss:.4f}")
    print(f"Valid Loss     : {valid_result['loss']:.4f}")
    print(f"Valid Accuracy : {valid_result['accuracy']:.4f}")
    print(f"Valid Precision: {valid_result['precision']:.4f}")
    print(f"Valid Recall   : {valid_result['recall']:.4f}")
    print(f"Valid F1       : {valid_result['f1']:.4f}")
    print("Valid Confusion Matrix")
    print(confusion_matrix(valid_result["y_true"], valid_result["y_pred"]))

    if valid_result["f1"] > best_valid_f1:
        best_valid_f1 = valid_result["f1"]
        best_state_dict = {
            key: value.cpu().clone()
            for key, value in model.state_dict().items()
        }
        print("Best model updated.")

    print("=" * 70)

history_df = pd.DataFrame(history)
display(history_df)


Epoch 1/5 | Step 50/151 | Loss: 0.6417
Epoch 1/5 | Step 100/151 | Loss: 0.5452
Epoch 1/5 | Step 150/151 | Loss: 0.1330
Epoch 1/5
Train Loss     : 0.3858
Valid Loss     : 0.4821
Valid Accuracy : 0.8307
Valid Precision: 0.9130
Valid Recall   : 0.3387
Valid F1       : 0.4941
Valid Confusion Matrix
[[190   2]
 [ 41  21]]
Best model updated.
Epoch 2/5 | Step 50/151 | Loss: 0.0383
Epoch 2/5 | Step 100/151 | Loss: 0.6745
Epoch 2/5 | Step 150/151 | Loss: 0.2485
Epoch 2/5
Train Loss     : 0.1624
Valid Loss     : 0.4948
Valid Accuracy : 0.8346
Valid Precision: 0.7778
Valid Recall   : 0.4516
Valid F1       : 0.5714
Valid Confusion Matrix
[[184   8]
 [ 34  28]]
Best model updated.
Epoch 3/5 | Step 50/151 | Loss: 0.0886
Epoch 3/5 | Step 100/151 | Loss: 0.0790
Epoch 3/5 | Step 150/151 | Loss: 0.3173
Epoch 3/5
Train Loss     : 0.0649
Valid Loss     : 0.7199
Valid Accuracy : 0.8386
Valid Precision: 0.8387
Valid Recall   : 0.4194
Valid F1       : 0.5591
Valid Confusion Matrix
[[187   5]
 [ 36  26]]
Epo

,epoch,train_loss,valid_loss,valid_accuracy,valid_precision,valid_recall,valid_f1
0,1,0.385827,0.482089,0.830709,0.913043,0.338710,0.494118
1,2,0.162356,0.494832,0.834646,0.777778,0.451613,0.571429
2,3,0.064862,0.719904,0.838583,0.838710,0.419355,0.559140
3,4,0.026340,0.559270,0.877953,0.860465,0.596774,0.704762
4,5,0.020660,0.580461,0.877953,0.816327,0.645161,0.720721


## 10. Test 평가

In [19]:
if best_state_dict is not None:
    model.load_state_dict({
        key: value.to(device)
        for key, value in best_state_dict.items()
    })

test_result = evaluate_model(model, test_loader, loss_fn, threshold=THRESHOLD)

print("=" * 80)
print("TEST RESULT")
print("=" * 80)
print(f"Accuracy : {test_result['accuracy']:.4f}")
print(f"Precision: {test_result['precision']:.4f}")
print(f"Recall   : {test_result['recall']:.4f}")
print(f"F1-score : {test_result['f1']:.4f}")

print("Classification Report")
print(classification_report(
    test_result["y_true"],
    test_result["y_pred"],
    target_names=["non-hate(0)", "hate(1)"],
    digits=4
))

print("Confusion Matrix")
print(confusion_matrix(test_result["y_true"], test_result["y_pred"]))


TEST RESULT
Accuracy : 0.8432
Precision: 0.6742
Recall   : 0.7143
F1-score : 0.6936
Classification Report
              precision    recall  f1-score   support

 non-hate(0)     0.9036    0.8858    0.8946       254
     hate(1)     0.6742    0.7143    0.6936        84

    accuracy                         0.8432       338
   macro avg     0.7889    0.8001    0.7941       338
weighted avg     0.8466    0.8432    0.8447       338

Confusion Matrix
[[225  29]
 [ 24  60]]


## 11. 결과 저장

In [20]:
proposal_result = {
    "model": OUTPUT_NAME,
    "split": "context_group_split",
    "input": "context + comment",
    "max_length": MAX_LENGTH,
    "epochs": EPOCHS,
    "lr": LR,
    "class_weight": "[1.0, 1.3]",
    "threshold": THRESHOLD,
    "best_valid_f1": best_valid_f1,
    "accuracy": test_result["accuracy"],
    "precision": test_result["precision"],
    "recall": test_result["recall"],
    "f1": test_result["f1"]
}

result_df = pd.DataFrame([proposal_result])
display(result_df)

result_df.to_csv("/content/proposal_result_bert_best.csv", index=False, encoding="utf-8-sig")
history_df.to_csv("/content/training_history_bert_best.csv", index=False, encoding="utf-8-sig")

print("저장 완료")
print("/content/proposal_result_bert_best.csv")
print("/content/training_history_bert_best.csv")


,model,split,input,max_length,epochs,lr,class_weight,threshold,best_valid_f1,accuracy,precision,recall,f1
0,BERT_context+comment_group_split_weak_class_we...,context_group_split,context + comment,256,5,0.00002,"[1.0, 1.3]",0.5,0.720721,0.843195,0.674157,0.714286,0.693642


저장 완료
/content/proposal_result_bert_best.csv
/content/training_history_bert_best.csv
